# Day 39 — Practice Questions

## Decision Trees

**Boston University · Data Science Program**

---

15 questions in four parts.

| part | topic | questions |
|---|---|---|
| **A** | Reading a tree | 4 |
| **B** | How a split is chosen | 4 |
| **C** | Overfitting and pruning | 4 |
| **D** | Judgement | 3 |

Run the setup cell first. Every answer is a number or a short sentence you can check by running code — if you are not sure, compute it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV)

CV = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- the same 900 scholarship decisions as the lesson -------------------
rng = np.random.default_rng(3900)
N = 900
marks         = rng.normal(70, 14, N).clip(35, 100).round(0)
family_income = rng.normal(380000, 210000, N).clip(40000, 1500000).round(0)
attendance    = rng.normal(82, 12, N).clip(40, 100).round(0)
sport         = rng.binomial(1, 0.25, N).astype(float)
rule = (((marks >= 75) & (attendance >= 80)) | (marks >= 88)
        | ((sport == 1) & (marks >= 62) & (attendance >= 75)))
rule = rule & (family_income <= 700000)
flip = rng.uniform(0, 1, N) < 0.08
awarded = np.where(flip, 1 - rule.astype(int), rule.astype(int))

sch = pd.DataFrame({"marks": marks, "family_income": family_income,
                    "attendance": attendance, "sport": sport, "awarded": awarded})
X = sch[["marks", "family_income", "attendance", "sport"]]
y = sch["awarded"]

def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    p = labels.mean()
    return 1 - p**2 - (1 - p)**2

print("setup ok —", len(sch), "rows, baseline", round(1 - y.mean(), 4))

---
# Part A — Reading a tree

### A1
Fit a `DecisionTreeClassifier` with `max_depth=2` and `random_state=0` on all of `X, y`,
and print it with `export_text`.

1. What question is at the root?
2. How many leaves does it have?
3. What fraction of the 900 students land in the biggest leaf?
4. Look at the two leaves under `marks <= 74.5`. What is odd about them, and what does
   that tell you about the split above them?

In [ ]:
# your code here

### A2
Here is a new applicant:

| marks | family_income | attendance | sport |
|---|---|---|---|
| 66 | 180000 | 91 | 1 |

Fit a **depth-3** tree. Before running `predict`, walk this student down the tree by hand
using the `export_text` output and write down which leaf you expect.

Then check with `predict` and `predict_proba`. Do they match?

In [ ]:
# your code here

### A3
Confirm that `predict_proba` really is just a count. Use `tree.apply()` to find which
leaf the A2 student landed in, pull out every *training* student in that same leaf, and
compute the fraction who were awarded yourself.

In [ ]:
# your code here

### A4
Multiply the `attendance` column by 100 and refit:

- a `DecisionTreeClassifier(max_depth=4, random_state=0)`
- a `KNeighborsClassifier(15)` with **no** scaler

Report the 5-fold CV accuracy of each, before and after. Explain the difference in one
sentence each.

In [ ]:
# your code here

---
# Part B — How a split is chosen

### B1
Take only the students who play a sport (`sport == 1`).

How many are there, what fraction were awarded, and what is the Gini impurity of that
group? Is this group easier or harder to predict than the full 900?

In [ ]:
# your code here

### B2
Score the split `attendance <= 85` the way the tree does: Gini on each side, weighted by
group size, then the drop from the parent's 0.4437.

Compare it with the root split `marks <= 74.5`. By how many times is the root better?

In [ ]:
# your code here

### B3
Find the *best* threshold on `attendance` by scanning every midpoint between consecutive
distinct values, exactly as the tree does.

How many candidates are there, which wins, and how does its impurity drop compare with
the root's 0.1068?

In [ ]:
# your code here

### B4
Why does the tree weight each side by how many students are in it, instead of just
averaging the two Gini values?

Construct a small example to show what goes wrong. Imagine a group of 100 students,
50 awarded and 50 not, and compare these two candidate splits:

- **Split A** — left: 3 students, all awarded. right: the other 97.
- **Split B** — left: 50 students, 45 awarded. right: 50 students, 5 awarded.

Compute both the **unweighted average** and the **weighted average** of the two Gini
values for each split, and say which split each method prefers.

In [ ]:
# your code here

---
# Part C — Overfitting and pruning

### C1
Build a table of `max_depth` in `[1, 2, 3, 4, 5, 6, 8, None]` showing, for each:
number of leaves, mean **training** accuracy, mean **cross-validated** accuracy, and the gap.

Use `cross_validate(..., return_train_score=True)` with `CV`.

Which depth is best? What is the gap at `None`, and what is it at the best depth?

In [ ]:
# your code here

### C2
Now hold depth unlimited and vary `min_samples_leaf` over `[1, 2, 5, 10, 20, 50, 100]`.

Which value gives the best CV score? Which gives the smallest gap? Are they the same
value, and what should you do when they disagree?

In [ ]:
# your code here

### C3
Use `GridSearchCV` to search `max_depth` and `min_samples_leaf` together.

How many models does it fit in total? What does it pick, and what does it score?

In [ ]:
# your code here

### C4
**The trap.** Suppose you had chosen `max_depth` by whichever value gave the highest
**training** accuracy, instead of the highest cross-validated accuracy.

Which depth would you have picked, what would it actually score on new data, and how much
accuracy would that mistake have cost you?

In [ ]:
# your code here

---
# Part D — Judgement

### D1
Day 39's headline was that the tree beat logistic regression by nearly 10 points. That
was a fact about **this data**, not about trees.

Here is a different problem: predicting whether a student passes an exam, from hours
studied and their previous score. This time the truth is a **smooth** relationship, not a
set of thresholds.

Run the cell below to build it, then compare: logistic regression, k-NN (k=15), a
depth-4 tree, and a tuned tree. Report the baseline too.

Which model wins? Is the winner clearly ahead?

In [ ]:
r2 = np.random.default_rng(777)
M = 900
hours      = r2.normal(12, 4, M).clip(0, 30).round(1)
prev_score = r2.normal(60, 15, M).clip(10, 100).round(0)

# a smooth logistic truth: no thresholds anywhere
z = -7.5 + 0.30 * hours + 0.075 * prev_score
passed = (r2.uniform(0, 1, M) < 1 / (1 + np.exp(-z))).astype(int)

exam = pd.DataFrame({"hours": hours, "prev_score": prev_score, "passed": passed})
XL, yL = exam[["hours", "prev_score"]], exam["passed"]
print(exam.head())
print("\npass rate:", round(yL.mean(), 4))

In [ ]:
# your code here

### D2
Fit a `max_depth=5` tree on `X, y` and print `feature_importances_`.

Then add a column `marks_again` that is an exact copy of `marks`, refit, and print the
importances again.

What happens to the importance of `marks`? What does that tell you about reading
feature importances when your columns are correlated?

In [ ]:
# your code here

### D3
Fit a depth-3 tree on the scholarship data and print it with `export_text`.

Now write the committee's rule **in plain English**, as you would put it in a letter to a
student. Then compare your version with the actual rule used to generate the data (it is
in the setup cell at the top). What did the tree get right, and what did it miss?

In [ ]:
# your code here

---
### End of questions

Check your answers against Day39_Practice_Solutions.ipynb once it is released. If a number does not match, re-run the setup cell first — every question depends on the same 900 rows.